# Day 35: Build an MCP Server

Every agent we've built has hardcoded tools. Change the agent? Rewrite the tools. Change the framework? Rewrite again.

**MCP (Model Context Protocol)** fixes this. It's a standard protocol between agents and tools — like USB for AI. Build a tool server once, any MCP-compatible agent can use it.

## Install

In [16]:
%pip install fastmcp langchain-mcp-adapters --quiet


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Why MCP?

Without MCP:
```
Agent A → custom tool code → Weather API
Agent B → different tool code → same Weather API
```

With MCP:
```
Agent A ─┐
         ├─ MCP Protocol ─→ Weather MCP Server → Weather API
Agent B ─┘
```

Build the server once. Any agent connects via the protocol.

## Build the Server

We'll build a **personal notes** MCP server — add notes, search them, list them. Three tools, exposed via protocol.

In [ ]:
%%writefile notes_server.py
import json
from pathlib import Path
from fastmcp import FastMCP

mcp = FastMCP("Notes")

NOTES_FILE = Path("notes_data.json")

def _load() -> dict[str, str]:
    if NOTES_FILE.exists():
        return json.loads(NOTES_FILE.read_text())
    return {}

def _save(notes: dict[str, str]):
    NOTES_FILE.write_text(json.dumps(notes, indent=2))

@mcp.tool()
def add_note(title: str, content: str) -> str:
    """Add a note with a title and content."""
    notes = _load()
    notes[title] = content
    _save(notes)
    return f"Note '{title}' saved."

@mcp.tool()
def search_notes(query: str) -> str:
    """Search notes by keyword. Returns matching notes."""
    notes = _load()
    results = [
        f"{title}: {content}"
        for title, content in notes.items()
        if query.lower() in title.lower() or query.lower() in content.lower()
    ]
    return "\n".join(results) if results else "No notes found."

@mcp.tool()
def list_notes() -> str:
    """List all saved note titles."""
    notes = _load()
    if not notes:
        return "No notes yet."
    return ", ".join(notes.keys())

if __name__ == "__main__":
    mcp.run(transport="stdio")

## What Just Happened?

- `FastMCP("Notes")` — creates an MCP server named "Notes"
- `@mcp.tool()` — exposes a function as an MCP tool (any client can discover and call it)
- `mcp.run(transport="stdio")` — starts the server

Three decorators. One server. Any MCP client can now discover these tools automatically.

## Transports: stdio vs http

MCP supports two transport methods:

| Transport | How it works | Use case |
|-----------|-------------|----------|
| **stdio** | Client launches server as subprocess, talks via stdin/stdout | Local tools, dev/testing |
| **http** | Server runs on a URL, client connects over network | Remote/shared servers, production |

```python
# stdio — local (what we use today)
mcp.run(transport="stdio")

# http — remote (for shared/production servers)
mcp.run(transport="streamable-http", host="0.0.0.0", port=8000)
```

We use `stdio` because it's simpler for notebooks — no port management, no background server.

## Connect and Test

`MultiServerMCPClient` connects to MCP servers and discovers their tools.

In [ ]:
from pathlib import Path
from langchain_mcp_adapters.client import MultiServerMCPClient

# Clean any leftover data from previous runs
Path("notes_data.json").unlink(missing_ok=True)

client = MultiServerMCPClient({
    "notes": {
        "transport": "stdio",
        "command": "python",
        "args": ["notes_server.py"],
    }
})

tools = await client.get_tools()
print(f"✅ Discovered {len(tools)} tools:")
for t in tools:
    print(f"  - {t.name}: {t.description}")

## Test the Tools

In [19]:
add_tool = next(t for t in tools if t.name == "add_note")
search_tool = next(t for t in tools if t.name == "search_notes")
list_tool = next(t for t in tools if t.name == "list_notes")

print(await add_tool.ainvoke({"title": "Python tip", "content": "Use list comprehensions for cleaner code"}))
print(await add_tool.ainvoke({"title": "Meeting note", "content": "Project deadline moved to Friday"}))

[{'type': 'text', 'text': "Note 'Python tip' saved.", 'id': 'lc_052200f9-12ac-4ed0-963b-90d73f03dfff'}]
[{'type': 'text', 'text': "Note 'Meeting note' saved.", 'id': 'lc_78e108f7-22bc-48eb-ab15-499d44abd5eb'}]


In [20]:
print("All notes:", await list_tool.ainvoke({}))
print("Search 'Python':", await search_tool.ainvoke({"query": "Python"}))
print("Search 'deadline':", await search_tool.ainvoke({"query": "deadline"}))

All notes: [{'type': 'text', 'text': 'No notes yet.', 'id': 'lc_2fa336c0-c8df-4c76-815b-4aa9a83e9ad1'}]
Search 'Python': [{'type': 'text', 'text': 'No notes found.', 'id': 'lc_7c1da9b0-002d-499d-b7f1-4a9e7fe33134'}]
Search 'deadline': [{'type': 'text', 'text': 'No notes found.', 'id': 'lc_efd24b12-bee6-4d0f-be4e-98ccb72d1158'}]


## Key Takeaways

1. **`FastMCP` + `@mcp.tool()`** — three decorators turn any Python function into an MCP-discoverable tool
2. **`stdio` for local, `http` for remote** — two transports, same protocol, same tools
3. **`client.get_tools()` discovers everything** — no hardcoding, the client asks the server what tools exist

Next: Day 36 — we plug these MCP tools into a LangGraph agent.